In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DO BANCO DE DADOS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# ==============================================================================
# 2. DETECTOR À PROVA DE FALHAS (Acha a verdadeira coluna de tempo)
# ==============================================================================
def extrair_minutos(val):
    if pd.isna(val): return np.nan
    s = str(val).lower().strip()
    if s in ['nan', 'none', '', 'unchecked', 'false', '0', 'não se aplica', 'nao se aplica']: return np.nan

    if ':' in s:
        try:
            p = s.split(':')
            return int(p[0]) * 60 + int(p[1])
        except: pass

    nums = re.findall(r'\d+', s)
    if not nums: return np.nan
    v = float(nums[0])

    if ('h' in s or 'hora' in s) and v <= 5: return v * 60
    return v

cols_candidatas = [c for c in df.columns if any(p in c.lower() for p in ['tempo', 'dura', 'minutos', 'atendimento']) and 'especialidade' not in c.lower()]

melhor_coluna = None
max_validos = -1

# Testa todas as colunas para ver qual realmente guarda os minutos das consultas
for c in cols_candidatas:
    qtd_validos = df[c].apply(extrair_minutos).notna().sum()
    if qtd_validos > max_validos:
        max_validos = qtd_validos
        melhor_coluna = c

col_tempo = melhor_coluna
cols_espec = [c for c in df.columns if 'especialidade' in c.lower()]
cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]
col_ciap = cols_ciap[0] if cols_ciap else None

if not col_tempo or max_validos == 0:
    raise ValueError("Falha Crítica: Nenhuma coluna de tempo com números reais foi encontrada.")

print("="*80)
print("🔍 INTELIGÊNCIA ARTIFICIAL: DIAGNÓSTICO DE COLUNAS")
print(f" -> A verdadeira coluna de Tempo é: '{col_tempo}'")
print(f" -> Ela possui {max_validos} consultas com o tempo anotado em números.")
print("="*80)

# ==============================================================================
# 3. EXTRAÇÃO DE DADOS E INDICADOR DE QUALIDADE
# ==============================================================================
df['Tempo_Minutos'] = df[col_tempo].apply(extrair_minutos)

# Isola os eventos de consulta para o cálculo de porcentagem ser justo
df_consultas = df[(df['Repeat Instrument'].notna()) | (df['Tempo_Minutos'].notna())].copy()
if df_consultas.empty: df_consultas = df.copy()

total_consultas = len(df_consultas)
qtd_preenchidos = df_consultas['Tempo_Minutos'].notna().sum()
qtd_vazios = total_consultas - qtd_preenchidos

perc_preenchidos = (qtd_preenchidos / total_consultas) * 100 if total_consultas > 0 else 0
perc_vazios = (qtd_vazios / total_consultas) * 100 if total_consultas > 0 else 0

# Desconsidera os vazios (sem inventar médias)
df_valid = df_consultas.dropna(subset=['Tempo_Minutos']).copy()

# ==============================================================================
# 4. CRUZAMENTO DE ESPECIALIDADES (DADOS REAIS)
# ==============================================================================
def extrair_cruzamento(dataframe):
    regs = []
    val_pos = ['checked', 'sim', 'yes', '1', 'true', 'verdadeiro']
    val_neg = ['nan', 'none', '', 'unchecked', 'false', '0', 'não se aplica', 'nao se aplica', 'não', 'nao']

    for idx, row in dataframe.iterrows():
        t = row['Tempo_Minutos']
        ciap = str(row[col_ciap]) if col_ciap else "Desconhecido"

        for c in cols_espec:
            v = str(row[c]).strip().lower()
            if pd.isna(row[c]) or v in val_neg: continue

            if 'choice=' in c.lower():
                if v in val_pos:
                    esp = c.split('choice=')[-1].replace(')', '').strip()
                    regs.append({'Especialidade': esp, 'Tempo_Minutos': t, 'CIAP': ciap})
            else:
                regs.append({'Especialidade': str(row[c]).strip().title(), 'Tempo_Minutos': t, 'CIAP': ciap})
    return pd.DataFrame(regs).drop_duplicates()

df_tempo = extrair_cruzamento(df_valid)

# Plano B (Fallback): Se o cruzamento der vazio, significa que o tempo e a especialidade
# foram preenchidos em formulários separados. O robô aplica a "Cascata" automaticamente.
if df_tempo.empty:
    print("Aviso: Tempo e Especialidade estão em formulários separados. Ativando a Cascata de Dados...")
    df_fallback = df.sort_values(by=['Record ID', 'Repeat Instance']).copy()
    df_fallback[cols_espec] = df_fallback.groupby('Record ID')[cols_espec].ffill()
    df_valid_fallback = df_fallback.dropna(subset=['Tempo_Minutos'])
    df_tempo = extrair_cruzamento(df_valid_fallback)

if df_tempo.empty:
    raise ValueError("Erro Fatal: Não foi possível alinhar os dados.")

# ==============================================================================
# 5. MATEMÁTICA DOS OUTLIERS (IQR)
# ==============================================================================
Q1 = df_tempo['Tempo_Minutos'].quantile(0.25)
Q3 = df_tempo['Tempo_Minutos'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = max(1, Q1 - 1.5 * IQR)
limite_superior = Q3 + 1.5 * IQR

df_normal = df_tempo[(df_tempo['Tempo_Minutos'] >= limite_inferior) & (df_tempo['Tempo_Minutos'] <= limite_superior)]
df_outliers = df_tempo[(df_tempo['Tempo_Minutos'] < limite_inferior) | (df_tempo['Tempo_Minutos'] > limite_superior)]

media_suja = df_tempo['Tempo_Minutos'].mean()
media_limpa = df_normal['Tempo_Minutos'].mean()

# ==============================================================================
# 6. GERAÇÃO DO GRÁFICO (DADOS REAIS S/ OUTLIERS)
# ==============================================================================
top_espec = df_normal['Especialidade'].value_counts().nlargest(10).index
df_plot = df_normal[df_normal['Especialidade'].isin(top_espec)].copy()
ordem_espec = df_plot.groupby('Especialidade')['Tempo_Minutos'].median().sort_values(ascending=False).index

plt.figure(figsize=(14, 8))
sns.set_theme(style="whitegrid")
paleta = sns.color_palette("plasma_r", len(ordem_espec))

ax = sns.violinplot(data=df_plot, x='Tempo_Minutos', y='Especialidade', order=ordem_espec, palette=paleta, inner=None, alpha=0.5, linewidth=0.5, scale='width')
sns.stripplot(data=df_plot, x='Tempo_Minutos', y='Especialidade', order=ordem_espec, color='black', alpha=0.3, size=3.5, jitter=0.25)
sns.pointplot(data=df_plot, x='Tempo_Minutos', y='Especialidade', order=ordem_espec, estimator=np.mean, color='red', markers='D', linestyles='', errwidth=0, label='Média')

plt.title(f"Duração Média do Teleatendimento (Apenas Preenchidos | S/ Outliers)\nMédia Global Limpa: {media_limpa:.0f} min", fontsize=15, fontweight='bold', pad=15)
plt.xlabel("Tempo de Consulta (Minutos)", fontsize=13)
plt.ylabel("")

estatisticas = df_plot.groupby('Especialidade')['Tempo_Minutos'].agg(['count', 'median']).reindex(ordem_espec)
novos_labels = [f"{str(esp)[:25]}\n(n={int(row['count'])} | Med. {row['median']:.0f}m)" for esp, row in estatisticas.iterrows()]
ax.set_yticklabels(novos_labels)

plt.tight_layout()
plt.show()

# ==============================================================================
# 7. EXPORTAÇÃO SILENCIOSA DA TABELA ESTATÍSTICA (GRÁFICO)
# ==============================================================================
# Agrupa os dados exatos que formam o gráfico desenhado
df_tabela = df_plot.groupby('Especialidade')['Tempo_Minutos'].agg(
    Quantidade_Consultas='count',
    Tempo_Minimo='min',
    Tempo_Medio='mean',
    Tempo_Mediano='median',
    Tempo_Maximo='max',
    Desvio_Padrao='std'
).reset_index()

# Arredonda as métricas para 1 casa decimal
cols_tempo = ['Tempo_Minimo', 'Tempo_Medio', 'Tempo_Mediano', 'Tempo_Maximo', 'Desvio_Padrao']
df_tabela[cols_tempo] = df_tabela[cols_tempo].round(1)

# Ordena a tabela pela Mediana (do maior tempo para o menor) para bater visualmente com o gráfico
df_tabela = df_tabela.set_index('Especialidade').reindex(ordem_espec).reset_index()

nome_arquivo_tabela = 'Tabela_Estatistica_Tempo_Especialidade.csv'
df_tabela.to_csv(nome_arquivo_tabela, index=False, encoding='utf-8-sig')

# ==============================================================================
# 8. RELATÓRIOS EXECUTIVOS NO TERMINAL
# ==============================================================================
print("\n" + "="*80)
print("📊 INDICADOR DE QUALIDADE: PREENCHIMENTO DO TEMPO DE CONSULTA")
print("="*80)
print(f"Total de consultas analisadas: {total_consultas}")
print(f" -> ✅ Preenchidas e válidas para o gráfico: {qtd_preenchidos} ({perc_preenchidos:.1f}%)")
print(f" -> ❌ Deixadas em branco (Desconsideradas):   {qtd_vazios} ({perc_vazios:.1f}%)")

if perc_vazios > 20:
    print("\n ⚠️ ALERTA DE GESTÃO: Mais de 20% das consultas estão sem registro de tempo!")

print("\n" + "="*80)
print(f"🚨 AUDITORIA DE OUTLIERS (Casos fora do padrão normal de {limite_inferior:.0f}m a {limite_superior:.0f}m)")
print("="*80)
print(f" - Média Real (S/ Outliers): {media_limpa:.1f} minutos")
print(f" - Média Distorcida (C/ Outliers): {media_suja:.1f} minutos")
print(f"Total de casos absurdos ignorados no gráfico: {len(df_outliers)} casos\n")

if not df_outliers.empty:
    outliers_altos = df_outliers[df_outliers['Tempo_Minutos'] > limite_superior]
    if not outliers_altos.empty:
        print(f"📈 CONSULTAS EXTREMAMENTE LONGAS (> {limite_superior:.0f} min): {len(outliers_altos)} casos")
        print("  Top Especialidades Afetadas:")
        print("  " + "\n  ".join(outliers_altos['Especialidade'].value_counts().head(3).to_string().split('\n')))
        print("\n  Top CIAPs (Diagnósticos) que puxaram o tempo para cima:")
        print("  " + "\n  ".join(outliers_altos['CIAP'].value_counts().head(3).to_string().split('\n')))
        print("-" * 50)

print(f"\n✅ Tabela base do gráfico salva silenciosamente em: '{nome_arquivo_tabela}'.\n")